# 05. Group-based Train / Validation / Test Split

이 노트북에서는 `master_manifest.csv`를 **`original_audio` 단위**로 Train / Validation / Test에 분할한다.

핵심 원칙:

```text
같은 original_audio
├─ REAL 1곡
└─ 여러 FAKE 생성물

→ 모두 반드시 동일한 split에 배정
```

행 단위 랜덤 분할은 같은 원곡에서 파생된 REAL/FAKE를 서로 다른 split에 넣을 수 있으므로 사용하지 않는다.

분할 비율:
- Train: 약 70%
- Validation: 약 15%
- Test: 약 15%

또한 `genre`를 기준으로 stratification하여 Electronic / Rock / Pop의 비율이 split 간 크게 달라지지 않도록 한다.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

MASTER_PATH = PROJECT_ROOT / "data/metadata/master_manifest.csv"
GROUP_SPLIT_PATH = PROJECT_ROOT / "data/metadata/original_audio_split.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/metadata/master_manifest_with_split.csv"

RANDOM_STATE = 42

print("MASTER_PATH      :", MASTER_PATH)
print("GROUP_SPLIT_PATH :", GROUP_SPLIT_PATH)
print("OUTPUT_PATH      :", OUTPUT_PATH)
print("RANDOM_STATE     :", RANDOM_STATE)


## 1. Master Manifest 로드 및 기본 확인

이전 단계에서 생성한 `master_manifest.csv`가 예상 구조를 갖는지 확인한다.


In [ ]:
master = pd.read_csv(MASTER_PATH)

print("===== MASTER MANIFEST =====")
print("Rows                 :", len(master))
print("Unique original_audio:", master["original_audio"].nunique())

print("\nLabel distribution:")
print(master["label"].value_counts())

print("\nGenre distribution:")
print(master["genre"].value_counts())

display(master.head())


## 2. original_audio 단위 Group Table 생성

각 `original_audio`가 하나의 장르만 갖는지 확인하고, group-level table을 만든다.


In [ ]:
group_table = (
    master
    .groupby("original_audio")
    .agg(
        genre=("genre", "first"),
        genre_count=("genre", "nunique"),
        total_samples=("sample_id", "size"),
        real_count=("label", lambda x: (x == "REAL").sum()),
        fake_count=("label", lambda x: (x == "FAKE").sum()),
    )
    .reset_index()
)

print("Groups:", len(group_table))
print("Groups with genre_count != 1:", int((group_table["genre_count"] != 1).sum()))
print("Groups with real_count != 1 :", int((group_table["real_count"] != 1).sum()))

print("\nGroup-level genre distribution:")
print(group_table["genre"].value_counts())

display(group_table.head())


## 3. original_audio Group을 70 / 15 / 15로 분할

먼저 Train 70%와 Temporary 30%로 나누고, Temporary를 Validation / Test로 절반씩 나눈다.

두 단계 모두 `genre`를 기준으로 stratification한다.


In [ ]:
train_groups, temp_groups = train_test_split(
    group_table,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=group_table["genre"],
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_groups["genre"],
)

train_groups = train_groups.copy()
val_groups = val_groups.copy()
test_groups = test_groups.copy()

train_groups["split"] = "train"
val_groups["split"] = "val"
test_groups["split"] = "test"

group_split = pd.concat(
    [train_groups, val_groups, test_groups],
    ignore_index=True,
)

print("===== GROUP SPLIT COUNTS =====")
print(group_split["split"].value_counts())

print("\nTotal groups:", len(group_split))


## 4. Group-level 장르 분포 확인

split별 `original_audio` 장르 분포가 유사하게 유지되는지 확인한다.


In [ ]:
group_genre_count = pd.crosstab(
    group_split["split"],
    group_split["genre"],
)

group_genre_ratio = pd.crosstab(
    group_split["split"],
    group_split["genre"],
    normalize="index",
).round(4)

print("===== GROUP-LEVEL GENRE COUNT =====")
display(group_genre_count)

print("===== GROUP-LEVEL GENRE RATIO =====")
display(group_genre_ratio)


## 5. Master Manifest에 split 부여

`original_audio`를 기준으로 group split 정보를 3,458개 sample에 결합한다.


In [ ]:
split_map = group_split[["original_audio", "split"]].copy()

master_split = master.merge(
    split_map,
    on="original_audio",
    how="left",
    validate="many_to_one",
)

print("Rows:", len(master_split))
print("Missing split:", master_split["split"].isna().sum())

print("\nSample-level split distribution:")
print(master_split["split"].value_counts())

display(master_split.head())


## 6. Source-family Leakage 검증

Train / Validation / Test 사이에 동일한 `original_audio`가 하나라도 겹치면 안 된다.


In [ ]:
train_ids = set(group_split.loc[group_split["split"] == "train", "original_audio"])
val_ids = set(group_split.loc[group_split["split"] == "val", "original_audio"])
test_ids = set(group_split.loc[group_split["split"] == "test", "original_audio"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("===== ORIGINAL_AUDIO OVERLAP CHECK =====")
print("Train ∩ Val :", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Val ∩ Test  :", len(val_test_overlap))

all_group_ids = train_ids | val_ids | test_ids

print("\nAssigned original_audio:", len(all_group_ids))
print("Expected original_audio:", master["original_audio"].nunique())


## 7. Split별 REAL / FAKE 분포 확인

원곡 단위 분할이므로 각 split의 REAL 수는 해당 split의 `original_audio` group 수와 같다.
FAKE 수는 원곡별 생성기 구성 차이에 따라 정확히 70/15/15 비율이 아닐 수 있다.


In [ ]:
label_count = pd.crosstab(
    master_split["split"],
    master_split["label"],
)

print("===== LABEL DISTRIBUTION BY SPLIT =====")
display(label_count)

label_ratio = pd.crosstab(
    master_split["split"],
    master_split["label"],
    normalize="index",
).round(4)

print("===== LABEL RATIO BY SPLIT =====")
display(label_ratio)


## 8. Split별 장르 분포 확인

sample-level에서도 장르 분포가 지나치게 치우치지 않았는지 확인한다.


In [ ]:
sample_genre_count = pd.crosstab(
    master_split["split"],
    master_split["genre"],
)

sample_genre_ratio = pd.crosstab(
    master_split["split"],
    master_split["genre"],
    normalize="index",
).round(4)

print("===== SAMPLE-LEVEL GENRE COUNT =====")
display(sample_genre_count)

print("===== SAMPLE-LEVEL GENRE RATIO =====")
display(sample_genre_ratio)


## 9. Split별 AI Generator 분포 확인

REAL의 `generator`는 결측값이므로 제외하고, FAKE 데이터만 대상으로 생성기 분포를 확인한다.


In [ ]:
fake_only = master_split[master_split["label"] == "FAKE"].copy()

generator_count = pd.crosstab(
    fake_only["split"],
    fake_only["generator"],
)

print("===== GENERATOR DISTRIBUTION BY SPLIT =====")
display(generator_count)


## 10. 최종 Split QC

다음 조건을 모두 확인한다.

- 전체 296개 `original_audio`가 정확히 한 split에 배정됨
- split 간 `original_audio` overlap이 없음
- 모든 sample에 split이 존재함
- 전체 sample 수가 3,458개로 유지됨
- REAL 296개 / FAKE 3,162개가 유지됨


In [ ]:
qc_summary = pd.DataFrame({
    "check": [
        "total_groups",
        "train_groups",
        "val_groups",
        "test_groups",
        "train_val_overlap",
        "train_test_overlap",
        "val_test_overlap",
        "master_rows",
        "missing_split",
        "real_rows",
        "fake_rows",
    ],
    "value": [
        len(all_group_ids),
        len(train_ids),
        len(val_ids),
        len(test_ids),
        len(train_val_overlap),
        len(train_test_overlap),
        len(val_test_overlap),
        len(master_split),
        int(master_split["split"].isna().sum()),
        int((master_split["label"] == "REAL").sum()),
        int((master_split["label"] == "FAKE").sum()),
    ],
})

display(qc_summary)

core_qc_pass = (
    len(all_group_ids) == 296
    and len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
    and len(master_split) == 3458
    and int(master_split["split"].isna().sum()) == 0
    and int((master_split["label"] == "REAL").sum()) == 296
    and int((master_split["label"] == "FAKE").sum()) == 3162
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", core_qc_pass)


## 11. Split 결과 저장

두 개의 파일을 저장한다.

1. `original_audio_split.csv`
   - 296개 원곡의 split 배정표

2. `master_manifest_with_split.csv`
   - 기존 master manifest에 `split` 컬럼을 추가한 sample-level manifest


In [ ]:
if not core_qc_pass:
    raise RuntimeError(
        "Split Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요."
    )

GROUP_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

group_split[
    [
        "original_audio",
        "genre",
        "total_samples",
        "real_count",
        "fake_count",
        "split",
    ]
].sort_values(["split", "original_audio"]).to_csv(
    GROUP_SPLIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

master_split.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved group split :", GROUP_SPLIT_PATH)
print("Saved master split:", OUTPUT_PATH)
print("Rows              :", len(master_split))


## 다음 단계

Split 검증이 완료되면 다음 단계는 **EDA 및 10초 segment 생성**이다.

이후 모든 모델 실험에서 반드시 동일한 split 파일을 재사용한다.

```text
master_manifest_with_split.csv
        ↓
EDA
        ↓
10초 segment manifest
        ↓
Logistic Regression / SVM
        ↓
Log-Mel CNN
```
